# 頭痛_遷移BERT強化版01：履歴あり遷移BERT（モデルA流用）／入力=閾値0.050データ

**強化版01の狙い**：B型遷移モデルに **履歴（前ノードの質問→答え）** を入れる効果を検証する。
モデルAは流用（一度計算したらキャッシュして再学習しない）。

| モデル | 学習 | 入力 |
|---|---|---|
| **モデルA（痛み/しびれ/振る舞い）** | 各ノード単体（painful の train_one_fold）。**結果はキャッシュ流用** | 採用ペア（質問なし） |
| **B0（履歴なし）** | 3ノード共有1本 | 質問 + 採用ペア |
| **B1（履歴あり）** | 3ノード共有1本 | **[これまでの確認] 前ノードQ→A** + 質問 + 採用ペア |

- 入力CSV：`dataset/input_pairs_threshold=0.050.csv`（135患者・採用ペアは cos_question 閾値0.050）。
- 学習時の履歴＝gold、ノード別評価＝gold履歴(teacher-forced)、トリアージ＝**予測履歴で逐次トラバース**。
- `BASE_MODEL` を ModernBERT-Ja 等に差し替えれば長文レバーも検証可。
- トリアージ/遷移/履歴の関数以外は `HeadacheBERT_painful_Finetuning.ipynb` 流用。

# 1. セットアップ

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/enenen13/Emergency_task'
REPO_NAME = 'Emergency_task'
REPO_BRANCH = 'feature/headache-ablation-notebook'
IN_COLAB = 'google.colab' in sys.modules

DATA_FILE = 'input_pairs_threshold=0.050.csv'   # ← 入力CSV（dataset/ 配下）


def _find_repo_root(start):
    d = os.path.abspath(start)
    for _ in range(6):
        if os.path.exists(os.path.join(d, 'dataset', DATA_FILE)):
            return d
        parent = os.path.dirname(d)
        if parent == d:
            break
        d = parent
    return None


REPO_DIR = _find_repo_root(os.getcwd())
cloned = False
if REPO_DIR is None:
    if not os.path.isdir(REPO_NAME):
        print(f'git clone -b {REPO_BRANCH} {REPO_URL} ...')
        subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL], check=True)
        cloned = True
    REPO_DIR = os.path.abspath(REPO_NAME)
os.chdir(REPO_DIR)
print('REPO_DIR =', REPO_DIR)

if IN_COLAB or cloned:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U',
                    'transformers', 'sentencepiece', 'fugashi', 'unidic-lite',
                    'accelerate', 'pyyaml'], check=True)

CSV_PATH = os.path.join(REPO_DIR, 'dataset', DATA_FILE)
YAML_PATH = os.path.join(REPO_DIR, 'transition_diagram', 'protocol.yaml')
OUT_DIR = os.path.join(REPO_DIR, 'output')
os.makedirs(OUT_DIR, exist_ok=True)
print('CSV :', CSV_PATH, '(exists:', os.path.exists(CSV_PATH), ')')

# 2. インポート & 設定

In [ ]:
import time, random, json
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import fugashi
import transformers
transformers.logging.set_verbosity_error()   # 余計なロードレポート/警告を抑制

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DEVICE =', DEVICE)

# 長文レバー：'sbintuitions/modernbert-ja-130m' などに差し替え可（MAX_LENGTHも伸ばすと効く）
BASE_MODEL = 'cl-tohoku/bert-base-japanese-v3'
MAX_LENGTH = 512
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
BATCH_SIZE = 8
N_FOLDS = 5
USE_STOPWORDS = False
NUM_LABELS = 3
SEED = 42
REUSE_MODEL_A = True   # True: モデルA予測をキャッシュ流用（無ければ計算して保存）


def set_seed(seed: int = SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()
print(f'BASE_MODEL={BASE_MODEL} MAX_LENGTH={MAX_LENGTH} epochs={NUM_EPOCHS} folds={N_FOLDS} reuseA={REUSE_MODEL_A}')

# 3. データ・決定木・採用ペア

閾値0.050データの採用ペア列：`採用ペア_cos_question_sudden`=痛み / `_numbness`=しびれ / `_behavior`=振る舞い。

In [ ]:
df = pd.read_csv(CSV_PATH)
print('rows:', len(df), '/ patients:', df['id'].nunique())

import yaml
_proto = yaml.safe_load(open(YAML_PATH, encoding='utf-8'))
_h = next(p for p in _proto['protocols'] if p['id'] == 'headache')
fallback_triage = _h['fallback']['if_all_symptom_questions_negative']


def _is_branch(n):
    return 'choices' in n and not n.get('metadata_only', False)


def _parse_choice(c):
    if c.get('triage'):
        return {'action': 'terminal', 'triage': c['triage']}
    if c.get('next'):
        return {'action': 'next', 'next_id': c['next']}
    return {'action': 'fallback', 'triage': fallback_triage}


branch_table = [{'id': n['id'], 'question': n['question'],
                 'choices': [_parse_choice(c) for c in n['choices']]}
                for n in _h['nodes'] if _is_branch(n)]
branch_ids = {b['id'] for b in branch_table}
triage_decode = {0: 'R3', 1: 'R2', 2: 'Y2'}
LABEL_TEXT = {0: 'はい', 1: 'いいえ', 2: '不明'}

# 採用ペア列は閾値0.050データの命名に合わせる
NODES = [
    {'key': '痛み',   'adopt': '採用ペア_cos_question_sudden',   'label': '痛み',   'bid': 'headache_sudden_severe'},
    {'key': 'しびれ', 'adopt': '採用ペア_cos_question_numbness', 'label': 'しびれ', 'bid': 'headache_numbness_paralysis'},
    {'key': '振る舞い', 'adopt': '採用ペア_cos_question_behavior', 'label': '振る舞い', 'bid': 'headache_abnormal_behavior'},
]
_qmap = {b['id']: b['question'] for b in branch_table}
for n in NODES:
    n['question'] = _qmap[n['bid']]
    assert n['adopt'] in df.columns, f"列が無い: {n['adopt']}"
NODE_BY_BID = {n['bid']: n for n in NODES}
print('branch_table:', [b['id'] for b in branch_table], '| fallback:', fallback_triage)

_tagger = fugashi.Tagger()
STOPWORD_EXTRA_WORDS = set(['の', 'は', 'を', 'に', 'が', 'で', 'と', 'も', 'から', 'より',
                            'へ', 'や', 'など', 'ので', 'けど', 'けれど', '、', '。', 'です', 'ます'])


def remove_stopwords(text: str) -> str:
    return ''.join(w.surface for w in _tagger(text) if w.surface not in STOPWORD_EXTRA_WORDS)


_adopt_cache = {}
def adopted_text(pid, node):
    key = (pid, node['key'])
    if key not in _adopt_cache:
        g = df[df['id'] == pid]
        ad = g[g[node['adopt']] == True]
        t = ' '.join(ad['ペア'].astype(str).tolist()) if len(ad) else '(発話なし)'
        _adopt_cache[key] = remove_stopwords(t) if USE_STOPWORDS else t
    return _adopt_cache[key]

# 4. painful 流用：学習・モデル部品（A は train_one_fold、B は train_b_model）

In [ ]:
_tok_cache: Dict[str, 'AutoTokenizer'] = {}
def get_tokenizer(name):
    if name not in _tok_cache:
        _tok_cache[name] = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    return _tok_cache[name]


class PainTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=MAX_LENGTH):
        self.texts, self.labels = list(texts), list(labels)
        self.tokenizer, self.max_length = tokenizer, max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(self.texts[idx], truncation=True, max_length=self.max_length,
                             padding='max_length', return_tensors='pt')
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item


def build_model(name):
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=NUM_LABELS,
                                                               trust_remote_code=True)
    tok = get_tokenizer(name)
    if len(tok) != model.config.vocab_size:
        model.resize_token_embeddings(len(tok))
    return model.to(DEVICE)


def train_one_fold(model_name, train_texts, train_labels, test_texts, test_labels,
                   learning_rate, num_epochs, batch_size, max_length=MAX_LENGTH, seed=SEED):
    """painful 流用。test_texts の順で y_pred を返す。"""
    set_seed(seed)
    tokenizer = get_tokenizer(model_name)
    model = build_model(model_name)
    train_loader = DataLoader(PainTextDataset(train_texts, train_labels, tokenizer, max_length),
                              batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(PainTextDataset(test_texts, test_labels, tokenizer, max_length),
                             batch_size=batch_size, shuffle=False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    model.train()
    for epoch in range(num_epochs):
        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            outputs = model(**batch)
            outputs.loss.backward()
            optimizer.step()
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in test_loader:
            batch.pop('labels')
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            all_preds += torch.argmax(model(**batch).logits, dim=-1).cpu().numpy().tolist()
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return {'y_pred': all_preds}


def train_b_model(texts, labels):
    set_seed(SEED)
    tok = get_tokenizer(BASE_MODEL)
    model = build_model(BASE_MODEL)
    loader = DataLoader(PainTextDataset(texts, labels, tok, MAX_LENGTH), batch_size=BATCH_SIZE, shuffle=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    model.train()
    for epoch in range(NUM_EPOCHS):
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()
            out = model(**batch)
            out.loss.backward()
            optimizer.step()
    model.eval()   # ★推論前にevalへ（dropoutを切る）
    return model, tok


@torch.no_grad()
def predict_one(model, tok, text):
    model.eval()   # 念のため（eval固定）
    enc = tok([text], truncation=True, max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    return int(model(**enc).logits.argmax(dim=-1).cpu())


print('学習部品を定義。')

# 5. 履歴・遷移の中核（新規）

In [ ]:
def render_history(history):
    if not history:
        return ''
    body = ' ; '.join(f'{q}→{a}' for q, a in history)
    return f'[これまでの確認] {body} '


def build_context(pid, node, history, use_history):
    head = render_history(history) if use_history else ''
    return head + node['question'] + ' ' + adopted_text(pid, node)


def build_train_examples(train_ids, use_history):
    texts, labels = [], []
    for pid in train_ids:
        history = []
        for node in NODES:
            texts.append(build_context(pid, node, history, use_history))
            lab = true_node[node['key']][pid]
            labels.append(lab)
            history.append((node['question'], LABEL_TEXT[lab]))
    return texts, labels


def predict_nodes_teacherforced(model, tok, test_ids, use_history):
    pred = {n['key']: {} for n in NODES}
    for pid in test_ids:
        history = []
        for node in NODES:
            pred[node['key']][pid] = predict_one(model, tok, build_context(pid, node, history, use_history))
            history.append((node['question'], LABEL_TEXT[true_node[node['key']][pid]]))
    return pred


def predict_triage_traverse(model, tok, pid, use_history):
    history = []
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        node = NODE_BY_BID[b['id']]
        p = predict_one(model, tok, build_context(pid, node, history, use_history))
        ch = b['choices'][p]
        history.append((b['question'], LABEL_TEXT[p]))
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage']
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage


def predict_triage_from_table(pred_by_key, pid):
    """A のように各ノード予測が確定している場合の決定木トラバース。"""
    current = branch_table[0]['id']
    while True:
        b = next(x for x in branch_table if x['id'] == current)
        ch = b['choices'][pred_by_key[NODE_BY_BID[b['id']]['key']][pid]]
        if ch['action'] in ('terminal', 'fallback'):
            return ch['triage']
        if ch.get('next_id') in branch_ids:
            current = ch['next_id']
            continue
        return fallback_triage


print('履歴・遷移の中核を定義。')

# 6. fold分割・正解・サニティ

In [ ]:
patients = np.array(sorted(df['id'].unique()))


def _to_triage(v):
    # トリアージ列が 0/1/2 でも 'R2'/'R3'/'Y2' でも文字列コードに正規化
    s = str(v).strip()
    return triage_decode[int(s)] if s in ('0', '1', '2') else s


true_node = {n['key']: {p: int(df[df['id'] == p][n['label']].iloc[0]) for p in patients} for n in NODES}
true_triage = {p: _to_triage(df[df['id'] == p]['トリアージ'].iloc[0]) for p in patients}

triage_strat = np.array([true_triage[p] for p in patients])  # 文字列ラベルでもStratifiedKFoldは可
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
FOLDS = [(patients[tr], patients[te]) for tr, te in skf.split(patients, triage_strat)]
print(f'{N_FOLDS}-fold。test患者数:', [len(te) for _, te in FOLDS])

_gold = {n['key']: true_node[n['key']] for n in NODES}
_acc = np.mean([predict_triage_from_table(_gold, p) == true_triage[p] for p in patients])
print(f'[sanity] gold答えで辿ったトリアージ一致率 = {_acc:.3f}（1.000ならOK）')

# 7. モデルA（流用）：各ノード単体 painful。キャッシュがあればロード

In [ ]:
A_CACHE = os.path.join(OUT_DIR, f'modelA_pred__{DATA_FILE}.json')

if REUSE_MODEL_A and os.path.exists(A_CACHE):
    _raw = json.load(open(A_CACHE, encoding='utf-8'))
    predA = {k: {pid: int(v) for pid, v in d.items()} for k, d in _raw.items()}
    print('[reuse] モデルA予測をロード:', A_CACHE)
else:
    predA = {n['key']: {} for n in NODES}
    for n in NODES:
        texts_by_pid = {p: adopted_text(p, n) for p in patients}
        for i, (tr_ids, te_ids) in enumerate(FOLDS):
            res = train_one_fold(
                BASE_MODEL,
                [texts_by_pid[p] for p in tr_ids], [true_node[n['key']][p] for p in tr_ids],
                [texts_by_pid[p] for p in te_ids], [true_node[n['key']][p] for p in te_ids],
                LEARNING_RATE, NUM_EPOCHS, BATCH_SIZE, MAX_LENGTH)
            for pid, p in zip(te_ids, res['y_pred']):
                predA[n['key']][pid] = p
            print(f"  A[{n['key']}] fold{i} done")
    json.dump({k: {str(pid): int(v) for pid, v in d.items()} for k, d in predA.items()},
              open(A_CACHE, 'w', encoding='utf-8'), ensure_ascii=False)
    print('[save] モデルA予測を保存:', A_CACHE)

# 8. モデルB0（履歴なし）/ B1（履歴あり）を学習・評価

In [ ]:
def run_B(use_history, tag):
    predN = {n['key']: {} for n in NODES}
    predTri = {}
    for i, (tr_ids, te_ids) in enumerate(FOLDS):
        texts, labels = build_train_examples(tr_ids, use_history)
        model, tok = train_b_model(texts, labels)
        pn = predict_nodes_teacherforced(model, tok, te_ids, use_history)
        for n in NODES:
            predN[n['key']].update(pn[n['key']])
        for pid in te_ids:
            predTri[pid] = predict_triage_traverse(model, tok, pid, use_history)
        print(f'  [{tag}] fold{i} done')
        del model
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return predN, predTri


print('=== B0（履歴なし）===')
predN0, predTri0 = run_B(False, 'B0')
print('=== B1（履歴あり）===')
predN1, predTri1 = run_B(True, 'B1')
print('完了')

# 9. 結果①：ノード別 accuracy / macro-F1（A vs B0 vs B1）

In [ ]:
def node_scores(pred_for_node, key):
    yt = [true_node[key][p] for p in patients]
    yp = [pred_for_node[p] for p in patients]
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


rows = []
for n in NODES:
    k = n['key']
    aA, fA = node_scores(predA[k], k)
    a0, f0 = node_scores(predN0[k], k)
    a1, f1v = node_scores(predN1[k], k)
    rows.append({'ノード': k,
                 'A acc': f'{aA:.3f}', 'B0 acc': f'{a0:.3f}', 'B1 acc': f'{a1:.3f}',
                 'A F1': f'{fA:.3f}', 'B0 F1': f'{f0:.3f}', 'B1 F1': f'{f1v:.3f}'})
node_table = pd.DataFrame(rows)
print('===== ノード別（A=単体painful / B0=履歴なし / B1=履歴あり）=====')
display(node_table)
node_table.to_csv(os.path.join(OUT_DIR, 'kyoka01_node.csv'), index=False, encoding='utf-8-sig')

# 10. 結果②：最終トリアージ（決定木を辿る）A vs B0 vs B1

In [ ]:
def triage_scores_table(pred_by_key):
    yt = [true_triage[p] for p in patients]
    yp = [predict_triage_from_table(pred_by_key, p) for p in patients]
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


def triage_scores_dict(predTri):
    yt = [true_triage[p] for p in patients]
    yp = [predTri[p] for p in patients]
    return accuracy_score(yt, yp), f1_score(yt, yp, average='macro', zero_division=0)


aA, fA = triage_scores_table(predA)
a0, f0 = triage_scores_dict(predTri0)
a1, f1v = triage_scores_dict(predTri1)
tri_table = pd.DataFrame([
    {'モデル': 'A 単体(痛み/しびれ/振る舞い)', 'トリアージ acc': f'{aA:.3f}', 'トリアージ macro-F1': f'{fA:.3f}'},
    {'モデル': 'B0 遷移(履歴なし)', 'トリアージ acc': f'{a0:.3f}', 'トリアージ macro-F1': f'{f0:.3f}'},
    {'モデル': 'B1 遷移(履歴あり)', 'トリアージ acc': f'{a1:.3f}', 'トリアージ macro-F1': f'{f1v:.3f}'},
])
print('===== 最終トリアージ R3/R2/Y2 =====')
display(tri_table)
tri_table.to_csv(os.path.join(OUT_DIR, 'kyoka01_triage.csv'), index=False, encoding='utf-8-sig')
print('saved: kyoka01_node.csv, kyoka01_triage.csv')

# 11. まとめ

- **A vs B0**：単体painful vs 遷移共有。**B0 vs B1**：履歴の効果（Δが+なら有効）。
- ノード別はgold履歴(teacher-forced)、トリアージは予測履歴で逐次トラバース（Aは各ノード予測で辿る）。
- 入力=閾値0.050（135患者）。モデルAは `output/modelA_pred__input_pairs_threshold=0.050.csv.json` にキャッシュ流用。
- `BASE_MODEL` をModernBERT-Jaに変えれば長文レバーも検証可。学習ループ等はpainful流用、履歴・遷移のみ新規。